In [1]:
import requests
import sqlite3
import pandas as pd
from IPython.display import display

API_URL = "https://openlibrary.org/search.json"
DB_NAME = "books.db"
BOOK_LIMIT = 100

In [2]:
params = {
    "title": "The Hobbit",
    "limit": BOOK_LIMIT
}

response = requests.get(API_URL, params=params, timeout=20)
response.raise_for_status()

data = response.json()
books = data.get("docs", [])

print("Total matching records reported by API:", data.get("numFound"))
print("Records fetched in this request:", len(books))

Total matching records reported by API: 224
Records fetched in this request: 100


In [3]:
if books:
    display(pd.json_normalize(books[:1]).T)
else:
    print("No records returned.")

,0
author_key,[OL26320A]
author_name,[J.R.R. Tolkien]
cover_edition_key,OL51711263M
cover_height,611
cover_i,14627509
cover_width,368
ebook_access,borrowable
edition_count,482
first_publish_year,1937
has_fulltext,True


In [4]:
book_records = []

for book in books:
    title = book.get("title")
    authors = book.get("author_name") or []
    year = book.get("first_publish_year")
    record_id = book.get("key")

    if not title or not record_id:
        continue

    book_records.append({
        "record_id": record_id.strip(),
        "title": title.strip(),
        "author": ", ".join(authors) if authors else "Unknown",
        "publication_year": year
    })

books_df = pd.DataFrame(book_records)

books_df = books_df.drop_duplicates(subset="record_id")
books_df = books_df.sort_values("title", na_position="last").reset_index(drop=True)


print("Cleaned records:", len(books_df))

Cleaned records: 100


In [5]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)

display(books_df)

,record_id,title,author,publication_year
0,/works/OL20493903W,A Brief History of the Hobbit,John D. Rateliff,2018.0
1,/works/OL16477282W,A reader's companion to The hobbit and the lord of the r...,"Joseph Mathewson, Isaac Asimov, C. S. Lewis, W. H. Auden...",1995.0
2,/works/OL28275670W,A-Z of JRR Tolkien's the Hobbit,Sarah Oliver,2012.0
3,/works/OL21034216W,Art of the Hobbit,"J.R.R. Tolkien, Christina Scull, Wayne G. Hammond",2011.0
4,/works/OL16637291W,Exploring J.R.R. Tolkien's The hobbit,Corey Olsen,2012.0
5,/works/OL25117415W,Finding God in the Hobbit,Jim Ware,2006.0
6,/works/OL5753335W,Finding God in the Hobbit,Jim Ware,2006.0
7,/works/OL34051627W,History of the Hobbit,J.R.R. Tolkien,2023.0
8,/works/OL21271107W,Hobbit : the Desolation of Smaug Chronicles,Weta,2013.0
9,/works/OL35889944W,Hobbit Quizzes : the Hobbit Facts and Trivia,Taulbee CHRISTEN,2021.0


In [6]:
with sqlite3.connect(DB_NAME) as conn:
    conn.execute("""
        CREATE TABLE IF NOT EXISTS books (
            record_id TEXT PRIMARY KEY,
            title TEXT NOT NULL,
            author TEXT,
            publication_year INTEGER
        )
    """)

print("SQLite table is ready.")

SQLite table is ready.


In [7]:
with sqlite3.connect(DB_NAME) as conn:
    for _, row in books_df.iterrows():
        conn.execute("""
            INSERT OR IGNORE INTO books
            (record_id, title, author, publication_year)
            VALUES (?, ?, ?, ?)
        """, (
            row["record_id"],
            row["title"],
            row["author"],
            row["publication_year"]
        ))

print("Cleaned records saved successfully.")

Cleaned records saved successfully.


In [8]:
with sqlite3.connect(DB_NAME) as conn:
    stored_df = pd.read_sql_query("""
        SELECT record_id, title, author, publication_year
        FROM books
        ORDER BY title COLLATE NOCASE
    """, conn)

display(stored_df)
print("Total records stored:", len(stored_df))

,record_id,title,author,publication_year
0,/works/OL20493903W,A Brief History of the Hobbit,John D. Rateliff,2018.0
1,/works/OL16477282W,A reader's companion to The hobbit and the lord of the r...,"Joseph Mathewson, Isaac Asimov, C. S. Lewis, W. H. Auden...",1995.0
2,/works/OL28275670W,A-Z of JRR Tolkien's the Hobbit,Sarah Oliver,2012.0
3,/works/OL21034216W,Art of the Hobbit,"J.R.R. Tolkien, Christina Scull, Wayne G. Hammond",2011.0
4,/works/OL16637291W,Exploring J.R.R. Tolkien's The hobbit,Corey Olsen,2012.0
5,/works/OL25117415W,Finding God in the Hobbit,Jim Ware,2006.0
6,/works/OL5753335W,Finding God in the Hobbit,Jim Ware,2006.0
7,/works/OL34051627W,History of the Hobbit,J.R.R. Tolkien,2023.0
8,/works/OL21271107W,Hobbit : the Desolation of Smaug Chronicles,Weta,2013.0
9,/works/OL35889944W,Hobbit Quizzes : the Hobbit Facts and Trivia,Taulbee CHRISTEN,2021.0


Total records stored: 100
